# All of Us Version 8 betas

In [8]:
import pandas as pd
from pathlib import Path
import math, re

PROJECT_ROOT = Path("../").resolve()
OUT_DIR = PROJECT_ROOT / "tracks" / "new_data"
INPUT_DIR = PROJECT_ROOT / "tracks" / "raw_emmer"

GENOME = "hg38"
chrom, start1, end1 = "chr19", 11_039_463, 11_163_820   # 1-based inclusive, like the browser

In [ ]:
EFFECTS_FILE = INPUT_DIR / "LDLR_3k_variants_aligned_aouv8_ukbb_partial.csv"
SEP = ","                                   # "," if it's a CSV
POS, REF, ALT, BETA = "hg38 Position", "Reference", "Alternate", "AOU_BETA"

df = pd.read_csv(EFFECTS_FILE, sep=SEP, usecols=[POS, REF, ALT, BETA], dtype={POS: str})

# Position is "11089462" (then it's on `chrom`) or "chr19:11089462"; assumed 1-based, as usual
pos_text = df[POS].str.strip()
has_chrom = pos_text.str.contains(":")
df["chrom"] = chrom
df.loc[has_chrom, "chrom"] = "chr" + pos_text[has_chrom].str.split(":").str[0].str.removeprefix("chr")
df["pos"] = pos_text.str.split(":").str[-1].str.replace(",", "").astype(int)
df = df[(df["chrom"] == chrom) & df["pos"].between(start1, end1)].copy()

# Express every beta as the effect of the NON-reference allele
flip = df[ALT].str.upper() == df[REF].str.upper()
df.loc[flip, BETA] = -df.loc[flip, BETA]
print(f"{len(df)} rows in region, {flip.sum()} flipped")

def write_effects(d, path, name):
    lines = [f'track type=bedGraph name="{name}"']
    lines += [f"{chrom}\t{p - 1}\t{p}\t{b:.4g}" for p, b in zip(d["pos"], d[BETA])]
    path.write_text("\n".join(lines) + "\n", newline="\n")

# Option A: one track, strongest effect per position (sign kept)
strongest = (df.assign(absb=df[BETA].abs())
               .sort_values("absb")
               .drop_duplicates("pos", keep="last")
               .sort_values("pos"))
write_effects(strongest, OUT_DIR / "aou_effects.bedGraph", "Effect")

# # Option B: one track per alternate base (useful for saturation-mutagenesis style data)
# for base in "ACGT":
#     sub = df[(df[ALT].str.upper() == base) & ~flip].sort_values("pos")
#     if len(sub):
#         write_effects(sub, OUT_DIR / f"effects_{base}.bedGraph", f"Effect {base}")

2328 rows in region, 0 flipped


In [ ]:
ref_len = df[REF].str.len()
alt_len = df[ALT].str.len()
# VCF-style indels share their first base ("anchor"); don't count it as affected
anchored = (ref_len != alt_len) & (df[REF].str[0] == df[ALT].str[0])
df["start"] = df["pos"] - 1 + anchored.astype(int)                    # 0-based
df["end"] = df["start"] + (ref_len - anchored.astype(int)).clip(lower=1)
df["kind"] = "snv"
df.loc[ref_len > alt_len, "kind"] = "del"
df.loc[ref_len < alt_len, "kind"] = "ins"
print(df["kind"].value_counts())

def write_effects(d, path, name):
    lines = [f'track type=bedGraph name="{name}"']
    lines += [f"{chrom}\t{s}\t{e}\t{b:.4g}" for s, e, b in zip(d["start"], d["end"], d[BETA])]
    path.write_text("\n".join(lines) + "\n", newline="\n")

# Option A: strongest effect per interval (an SNV and a deletion at the same spot both stay)
strongest = (df.assign(absb=df[BETA].abs())
               .sort_values("absb")
               .drop_duplicates(["start", "end"], keep="last")
               .sort_values(["start", "end"]))
# write_effects(strongest, OUT_DIR / "aou_effects.bedGraph", "Effect")

# # Option B: per alternate base (SNVs only), plus one track for indels
# for base in "ACGT":
#     sub = df[(df["kind"] == "snv") & (df[ALT].str.upper() == base) & ~flip].sort_values("start")
#     if len(sub):
#         write_effects(sub, OUT_DIR / f"effects_{base}.bedGraph", f"Effect {base}")
# indels = df[df["kind"] != "snv"].sort_values("start")
# if len(indels):
#     write_effects(indels, OUT_DIR / "effects_indels.bedGraph", "Effect indels")

kind
snv    2141
del     119
ins      68
Name: count, dtype: int64


In [13]:
P_THRESHOLD = 1e-5                                 # keep variants with p <= this
POS, REF, ALT, BETA = "hg38 Position", "Reference", "Alternate", "AOU_BETA"
RSID, AF, VEP, P = "rsIDs", "Allele Frequency", "VEP Annotation", "AOU_P"   # match your header
OUT_FILE = OUT_DIR / "aou_effects_peaks.bed"
POS_RGB, NEG_RGB = "44,109,181", "200,55,45"          # β > 0 blue, β < 0 red

def neg_log10_p(p_text):
    """-log10(p) from the text, so p-values too small for a float (e.g. '3e-400') still work."""
    s = str(p_text).strip()
    m = re.fullmatch(r"([0-9.]+)[eE]([-+]?\d+)", s)
    if m:
        return -(math.log10(float(m.group(1))) + int(m.group(2)))
    try:
        v = float(s)
    except ValueError:
        return float("nan")
    return -math.log10(v) if v > 0 else float("nan")

def text(v):
    """Clean optional text cell: '' for missing, '.', or NaN."""
    return "" if not isinstance(v, str) or v.strip() in ("", ".") else v.strip()

# ---- read, place on the genome, keep the region ---------------------------------
df = pd.read_csv(EFFECTS_FILE, sep=SEP, usecols=[POS, REF, ALT, BETA, RSID, AF, VEP, P],
                 dtype={POS: str, P: str, RSID: str, VEP: str})
# round the P column to 3 significant figures. It's currently a string which may be in scientific notation, e.g. "3e-400", which is too small for a float.
df[P] = df[P].apply(lambda x: f"{float(x):.3g}" if isinstance(x, str) and x.strip() else x)

pos_text = df[POS].str.strip()
has_chrom = pos_text.str.contains(":")
df["chrom"] = chrom
df.loc[has_chrom, "chrom"] = "chr" + pos_text[has_chrom].str.split(":").str[0].str.removeprefix("chr")
df["pos"] = pos_text.str.split(":").str[-1].str.replace(",", "").astype(int)   # 1-based
df = df[(df["chrom"] == chrom) & df["pos"].between(start1, end1)].copy()

# ---- p-value threshold -----------------------------------------------------------
df["mlog10p"] = df[P].map(neg_log10_p)
df = df[df["mlog10p"] >= -math.log10(P_THRESHOLD)].copy()

# ---- express β and AF for the non-reference allele -------------------------------
flip = df[ALT].str.upper() == df[REF].str.upper()
df.loc[flip, BETA] = -df.loc[flip, BETA]
df.loc[flip, AF] = 1 - df.loc[flip, AF]

# ---- spans: SNV = 1 base; VCF-style indels skip the shared anchor base -----------
ref_len, alt_len = df[REF].str.len(), df[ALT].str.len()
anchored = (ref_len != alt_len) & (df[REF].str[0] == df[ALT].str[0])
df["start"] = df["pos"] - 1 + anchored.astype(int)                          # 0-based
df["end"] = df["start"] + (ref_len - anchored.astype(int)).clip(lower=1)

# ---- write BED9, colored by direction of effect ------------------------------------
def label(r, flipped):
    change = f"{r[REF]} (effect = ref, flipped)" if flipped else f"{r[REF]}>{r[ALT]}"
    parts = [text(r[RSID]), change, f"β={r[BETA]:.3g}", f"AF={float(r[AF]):.3g}",
             f"p={r[P].strip()}", text(r[VEP])]
    return "  ".join(p for p in parts if p).replace("\t", " ")

lines = [f'track name="Variants (p ≤ {P_THRESHOLD:g})" itemRgb="On"']
for i, r in df.sort_values(["start", "end"]).iterrows():
    s, e = int(r["start"]), int(r["end"])
    rgb = POS_RGB if r[BETA] >= 0 else NEG_RGB
    lines.append(f"{chrom}\t{s}\t{e}\t{label(r, flip[i])}\t0\t.\t{s}\t{e}\t{rgb}")

# OUT_FILE.write_text("\n".join(lines) + "\n", encoding="utf-8", newline="\n")
# print(f"{len(df)} variants with p ≤ {P_THRESHOLD:g} written to {OUT_FILE.name}")

In [14]:
OUT_FILE1 = OUT_DIR / "aou_LDL_inc_effects_peaks.bed"
OUT_FILE2 = OUT_DIR / "aou_LDL_dec_effects_peaks.bed"
lines1 = [f'track name="Variants (p ≤ {P_THRESHOLD:g}, β > 0)" itemRgb="On"']
lines2 = [f'track name="Variants (p ≤ {P_THRESHOLD:g}, β < 0)" itemRgb="On"']
for i, r in df.sort_values(["start", "end"]).iterrows():
    s, e = int(r["start"]), int(r["end"])
    rgb = POS_RGB if r[BETA] >= 0 else NEG_RGB
    line = f"{chrom}\t{s}\t{e}\t{label(r, flip[i])}\t0\t.\t{s}\t{e}\t{rgb}"
    if r[BETA] >= 0:
        lines1.append(line)
    else:
        lines2.append(line)
OUT_FILE1.write_text("\n".join(lines1) + "\n", encoding="utf-8", newline="\n")
OUT_FILE2.write_text("\n".join(lines2) + "\n", encoding="utf-8", newline="\n")
print(f"{len(df)} variants with p ≤ {P_THRESHOLD:g} written to {OUT_FILE1.name} and {OUT_FILE2.name}")

289 variants with p ≤ 1e-05 written to aou_LDL_inc_effects_peaks.bed and aou_LDL_dec_effects_peaks.bed
